# 효과적으로 학습하기 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV

from scikeras.wrappers import KerasRegressor
from sklearn.metrics import make_scorer, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import mse
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from keras.callbacks import EarlyStopping


In [2]:
total_seoul_data = pd.read_csv('./Total_Data_Seoul.csv', engine = 'python')
total_seoul_data

,측정일시,SO2,CO,O3,NO2,PM10,Seoul_temp(°C),Seoul_Precipitation(mm),Seoul_Wind_Speed(m/s),Seoul_Humidity(%),Seoul_Dew_point(°C)
0,2020-01-01 01:00:00,0.002,0.5,0.011,0.024,19.0,-5.9,1.556,1.7,40.0,-17.3
1,2020-01-01 02:00:00,0.002,0.6,0.005,0.030,19.0,-5.7,1.556,0.1,42.0,-16.5
2,2020-01-01 03:00:00,0.002,0.6,0.002,0.033,27.0,-5.6,0.000,0.0,46.0,-15.4
3,2020-01-01 04:00:00,0.002,0.6,0.003,0.031,20.0,-5.4,1.556,0.0,50.0,-14.2
4,2020-01-01 05:00:00,0.002,0.7,0.003,0.031,21.0,-5.2,1.556,0.0,55.0,-12.8
...,...,...,...,...,...,...,...,...,...,...,...
8778,2020-12-31 19:00:00,0.002,0.4,0.016,0.023,26.0,-7.1,1.556,2.4,58.0,-13.9
8779,2020-12-31 20:00:00,0.002,0.4,0.014,0.026,29.0,-7.1,1.556,3.2,59.0,-13.7
8780,2020-12-31 21:00:00,0.002,0.4,0.017,0.021,23.0,-7.2,1.556,2.7,61.0,-13.4
8781,2020-12-31 22:00:00,0.002,0.4,0.025,0.013,28.0,-7.4,1.556,2.5,66.0,-12.6


In [3]:
total_seoul_data = total_seoul_data.iloc[:,1:]
total_seoul_data

,SO2,CO,O3,NO2,PM10,Seoul_temp(°C),Seoul_Precipitation(mm),Seoul_Wind_Speed(m/s),Seoul_Humidity(%),Seoul_Dew_point(°C)
0,0.002,0.5,0.011,0.024,19.0,-5.9,1.556,1.7,40.0,-17.3
1,0.002,0.6,0.005,0.030,19.0,-5.7,1.556,0.1,42.0,-16.5
2,0.002,0.6,0.002,0.033,27.0,-5.6,0.000,0.0,46.0,-15.4
3,0.002,0.6,0.003,0.031,20.0,-5.4,1.556,0.0,50.0,-14.2
4,0.002,0.7,0.003,0.031,21.0,-5.2,1.556,0.0,55.0,-12.8
...,...,...,...,...,...,...,...,...,...,...
8778,0.002,0.4,0.016,0.023,26.0,-7.1,1.556,2.4,58.0,-13.9
8779,0.002,0.4,0.014,0.026,29.0,-7.1,1.556,3.2,59.0,-13.7
8780,0.002,0.4,0.017,0.021,23.0,-7.2,1.556,2.7,61.0,-13.4
8781,0.002,0.4,0.025,0.013,28.0,-7.4,1.556,2.5,66.0,-12.6


In [4]:
def extract_inputoutput(dataframe, lookback_time=3, predict_time=1):
    dfx = pd.DataFrame()
    dfy = pd.DataFrame()

    for i in range(len(dataframe) - (lookback_time - 1) - (predict_time)):
        if i % 1000 == 0:
            print(i)
        rowx = []
        for timestep in range(lookback_time):
            dfRename = dataframe.iloc[[i+timestep]]
            dfRename.index=[i]
            rowx.append(dfRename)
        rowx = pd.concat(rowx, axis=1)
        dfx = pd.concat([dfx, rowx])
        rowy = []
        rowy = pd.DataFrame([dataframe['PM10'][i+lookback_time]])
        dfy = pd.concat([dfy, rowy], ignore_index=True)

    print('X, Y 데이터 분류 완료!')
    return dfx, dfy

In [5]:
x, t = extract_inputoutput(total_seoul_data)

0
1000
2000
3000
4000
5000
6000
7000
8000
X, Y 데이터 분류 완료!


In [6]:
x

,SO2,CO,O3,NO2,PM10,Seoul_temp(°C),Seoul_Precipitation(mm),Seoul_Wind_Speed(m/s),Seoul_Humidity(%),Seoul_Dew_point(°C),...,SO2,CO,O3,NO2,PM10,Seoul_temp(°C),Seoul_Precipitation(mm),Seoul_Wind_Speed(m/s),Seoul_Humidity(%),Seoul_Dew_point(°C)
0,0.002,0.5,0.011,0.024,19.0,-5.9,1.556,1.7,40.0,-17.3,...,0.002,0.6,0.002,0.033,27.0,-5.6,0.000,0.0,46.0,-15.4
1,0.002,0.6,0.005,0.030,19.0,-5.7,1.556,0.1,42.0,-16.5,...,0.002,0.6,0.003,0.031,20.0,-5.4,1.556,0.0,50.0,-14.2
2,0.002,0.6,0.002,0.033,27.0,-5.6,0.000,0.0,46.0,-15.4,...,0.002,0.7,0.003,0.031,21.0,-5.2,1.556,0.0,55.0,-12.8
3,0.002,0.6,0.003,0.031,20.0,-5.4,1.556,0.0,50.0,-14.2,...,0.002,0.7,0.002,0.032,23.0,-4.8,0.000,1.9,58.0,-11.8
4,0.002,0.7,0.003,0.031,21.0,-5.2,1.556,0.0,55.0,-12.8,...,0.002,0.7,0.002,0.032,22.0,-4.6,1.556,2.1,62.0,-10.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8775,0.002,0.3,0.031,0.008,20.0,-5.5,1.556,3.5,47.0,-15.0,...,0.002,0.3,0.027,0.011,24.0,-6.7,0.000,2.5,54.0,-14.4
8776,0.002,0.3,0.029,0.010,24.0,-6.1,1.556,1.8,50.0,-14.8,...,0.002,0.4,0.016,0.023,26.0,-7.1,1.556,2.4,58.0,-13.9
8777,0.002,0.3,0.027,0.011,24.0,-6.7,0.000,2.5,54.0,-14.4,...,0.002,0.4,0.014,0.026,29.0,-7.1,1.556,3.2,59.0,-13.7
8778,0.002,0.4,0.016,0.023,26.0,-7.1,1.556,2.4,58.0,-13.9,...,0.002,0.4,0.017,0.021,23.0,-7.2,1.556,2.7,61.0,-13.4


In [7]:
t

,0
0,20.0
1,21.0
2,23.0
3,22.0
4,21.0
...,...
8775,26.0
8776,29.0
8777,23.0
8778,28.0


In [8]:
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2, shuffle=False)

print('x_train shape : ', x_train.shape)
print('t_train shape : ', t_train.shape)
print('x_test shape : ', x_test.shape)
print('t_test shape : ', t_test.shape)

x_train shape :  (7024, 30)
t_train shape :  (7024, 1)
x_test shape :  (1756, 30)
t_test shape :  (1756, 1)


In [9]:
timesteps = 3
feature = 10

x_train = np.array(x_train)
x_train = x_train.reshape(x_train.shape[0], timesteps, feature)

x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], timesteps, feature)

t_train = np.array(t_train)
t_test = np.array(t_test)

print('reshape 후 x_train shape :', x_train.shape)
print('t_train shape : ', t_train.shape)
print('reshape 후 x_test shape :', x_test.shape)
print('t_test shape : ', t_test.shape)

reshape 후 x_train shape : (7024, 3, 10)
t_train shape :  (7024, 1)
reshape 후 x_test shape : (1756, 3, 10)
t_test shape :  (1756, 1)


In [10]:
hyperparam_list = {'epochs' : [10, 20],
                   'batch_size' : [16, 32]}

print('사용할 hyperparameter 목록 : ', hyperparam_list)


사용할 hyperparameter 목록 :  {'epochs': [10, 20], 'batch_size': [16, 32]}


In [11]:
def lstm_model():
    cell_size = 128
    timesteps = 3
    feature = 10

    model = Sequential(name='AirPollution_LSTM')

    model.add(LSTM(cell_size, input_shape=(timesteps, feature), return_sequences=True))
    model.add(LSTM(cell_size))
    model.add(Dropout(0.1))

    model.add(Dense(1))
    model.compile(loss=mse, optimizer=Adam(learning_rate=0.001),
                  metrics=[RootMeanSquaredError()])
    
    return model

In [12]:
AirPollution_LSTM_model = KerasRegressor(model=lstm_model, verbose = 1)
grid_search = GridSearchCV(estimator=AirPollution_LSTM_model,
                           param_grid = hyperparam_list,
                           scoring = make_scorer(mean_squared_error, greater_is_better=False),
                           cv = 2)
print('grid search : ', grid_search, sep='\n')

grid search : 
GridSearchCV(cv=2,
             estimator=KerasRegressor(model=<function lstm_model at 0x124799080>),
             param_grid={'batch_size': [16, 32], 'epochs': [10, 20]},
             scoring=make_scorer(mean_squared_error, greater_is_better=False, response_method='predict'))


In [13]:
grid_search_result = grid_search.fit(x_train, t_train)

Epoch 1/10


/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 215.9644 - root_mean_squared_error: 14.6957
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 69.6227 - root_mean_squared_error: 8.3440
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50.9249 - root_mean_squared_error: 7.1362
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 43.3548 - root_mean_squared_error: 6.5844
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 41.2372 - root_mean_squared_error: 6.4216
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.3569 - root_mean_squared_error: 6.1933
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 37.4527 - root_mean_squared_error: 6.1199
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 36.8951 - root_mean_squared_error: 6.0741
Epoch 9/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 35.7596 - root_mean_squared_error: 5.9799
Epoch 10/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 34.2721 - root_mean_squared_er

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 766.9940 - root_mean_squared_error: 27.6947 
Epoch 2/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 309.0629 - root_mean_squared_error: 17.5802
Epoch 3/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 208.7619 - root_mean_squared_error: 14.4486
Epoch 4/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 167.1768 - root_mean_squared_error: 12.9297
Epoch 5/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 146.5633 - root_mean_squared_error: 12.1063
Epoch 6/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 133.0159 - root_mean_squared_error: 11.5333
Epoch 7/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 122.0749 - root_mean_squared_error: 11.0487
Epoch 8/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 116.7646 - root_mean_squared_error: 10.8058
Epoch 9/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 111.2227 - root_mean_squared_error: 10.5462
Epoch 10/10
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 103.6521 - ro

...

110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 282.7400 - root_mean_squared_error: 16.8149
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 104.1349 - root_mean_squared_error: 10.2047
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 67.2068 - root_mean_squared_error: 8.1980
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 54.9362 - root_mean_squared_error: 7.4119
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 47.0702 - root_mean_squared_error: 6.8608
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 44.7863 - root_mean_squared_error: 6.6923
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 42.0661 - root_mean_squared_error: 6.4858
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 39.7471 - root_mean_squared_error: 6.3045
Epoch 9/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.6035 - root_mean_squared_error: 6.2132
Epoch 10/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.2962 - root_mean_squared_

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


110/110 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 993.6452 - root_mean_squared_error: 31.5221 
Epoch 2/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 516.0148 - root_mean_squared_error: 22.7160
Epoch 3/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 340.8430 - root_mean_squared_error: 18.4619
Epoch 4/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 257.9790 - root_mean_squared_error: 16.0617
Epoch 5/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 214.5948 - root_mean_squared_error: 14.6491
Epoch 6/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 188.6433 - root_mean_squared_error: 13.7347
Epoch 7/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 167.7798 - root_mean_squared_error: 12.9530
Epoch 8/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 156.1685 - root_mean_squared_error: 12.4967
Epoch 9/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 146.7689 - root_mean_squared_error: 12.1148
Epoch 10/20
110/110 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 139.4407 - ro

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


439/439 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 325.5654 - root_mean_squared_error: 18.0434
Epoch 2/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 118.7529 - root_mean_squared_error: 10.8974
Epoch 3/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 91.8569 - root_mean_squared_error: 9.5842
Epoch 4/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 81.8527 - root_mean_squared_error: 9.0472
Epoch 5/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 74.2000 - root_mean_squared_error: 8.6139
Epoch 6/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 68.7059 - root_mean_squared_error: 8.2889
Epoch 7/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 65.8025 - root_mean_squared_error: 8.1119
Epoch 8/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 63.6934 - root_mean_squared_error: 7.9808
Epoch 9/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 59.9905 - root_mean_squared_error: 7.7454
Epoch 10/20
439/439 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 58.9982 - root_mean_squared_

In [14]:
print("Grid Search에서의 최고의 하이퍼파라미터 조합 : ", grid_search_result.best_params_)

Grid Search에서의 최고의 하이퍼파라미터 조합 :  {'batch_size': 16, 'epochs': 20}


In [15]:
air_pollution_lstm = lstm_model()
air_pollution_lstm.summary()

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "AirPollution_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_18 (LSTM)                  │ (None, 3, 128)         │        71,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_19 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,881 (792.50 KB)

 Trainable params: 202,881 (792.50 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
x_train, x_validation, t_train, t_validation = train_test_split(x_train, t_train, test_size=0.5, shuffle=False)

print('x_train shape : ', x_train.shape)
print('t_train shape : ', t_train.shape)
print('x_validation shape : ', x_validation.shape)
print('t_validation shape : ', t_validation.shape)

x_train shape :  (3512, 3, 10)
t_train shape :  (3512, 1)
x_validation shape :  (3512, 3, 10)
t_validation shape :  (3512, 1)


In [17]:
batch_size = grid_search.best_params_['batch_size']
epoch = grid_search.best_params_['epochs']

early_stopping = EarlyStopping(monitor='val_loss', mode='min', patience = epoch*0.3)
air_pollution_lstm.fit(x_train, t_train, validation_data=(x_validation, t_validation),
                       batch_size=batch_size, epochs = epoch, 
                       callbacks =[early_stopping])


Epoch 1/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 768.1229 - root_mean_squared_error: 27.7150 - val_loss: 88.4094 - val_root_mean_squared_error: 9.4026
Epoch 2/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 309.8906 - root_mean_squared_error: 17.6037 - val_loss: 46.6122 - val_root_mean_squared_error: 6.8273
Epoch 3/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 208.9772 - root_mean_squared_error: 14.4560 - val_loss: 39.6218 - val_root_mean_squared_error: 6.2946
Epoch 4/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 168.8084 - root_mean_squared_error: 12.9926 - val_loss: 39.2140 - val_root_mean_squared_error: 6.2621
Epoch 5/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 147.4893 - root_mean_squared_error: 12.1445 - val_loss: 36.9345 - val_root_mean_squared_error: 6.0774
Epoch 6/20
220/220 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 133.0931 - root_mean_squared_error: 11.5366 - val_loss: 33.9426 - val_root_mean_squared_error: 5.8260
Epoch 7/20
220/220 ━━━━━━━━━

In [18]:
pred = air_pollution_lstm.predict(x_test)

for i in range(1, 10):
    print('PM10 예측 : ', round(pred[i][0], 2), '/정답 : ', round(t_test[i][0], 2))

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
PM10 예측 :  77.95 /정답 :  75.0
PM10 예측 :  74.79 /정답 :  70.0
PM10 예측 :  68.06 /정답 :  74.0
PM10 예측 :  76.25 /정답 :  65.0
PM10 예측 :  64.36 /정답 :  59.0
PM10 예측 :  58.6 /정답 :  56.0
PM10 예측 :  54.12 /정답 :  52.0
PM10 예측 :  49.96 /정답 :  53.0
PM10 예측 :  51.07 /정답 :  51.0


In [19]:
loss, rmse = air_pollution_lstm.evaluate(x_test, t_test, verbose=1)
print('test loss(MSE) : ', round(loss, 6))
print('test RMSE : ', round(rmse, 2))

55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 965us/step - loss: 57.7474 - root_mean_squared_error: 7.5992
test loss(MSE) :  57.74736
test RMSE :  7.6


## 실습(2)

In [20]:
import tensorflow as tf
import os

In [21]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from scikeras.wrappers import KerasRegressor
from sklearn.metrics import make_scorer, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.losses import mse
from tensorflow.keras.metrics import RootMeanSquaredError, MeanSquaredError
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from keras.callbacks import EarlyStopping

In [22]:
zip_path = tf.keras.utils.get_file(
    origin = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip',
    fname = 'jena_climate_2009_2016.csv.zip',
    extract=True
)

In [23]:
csv_dir = os.path.splitext(zip_path)[0]
csv_path = os.path.join(csv_dir, 'jena_climate_2009_2016.csv')
print('파일 경로 : ', csv_path)

파일 경로 :  /Users/yuseunghwan/.keras/datasets/jena_climate_2009_2016_extracted/jena_climate_2009_2016.csv


In [24]:
weather_data = pd.read_csv(csv_path)
weather_data

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.30,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.40,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.90,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.20,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.10,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420546,31.12.2016 23:20:00,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
420547,31.12.2016 23:30:00,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
420548,31.12.2016 23:40:00,999.82,-3.16,270.01,-8.21,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2
420549,31.12.2016 23:50:00,999.81,-4.23,268.94,-8.53,71.80,4.46,3.20,1.26,1.99,3.20,1293.56,1.49,2.16,225.8


In [26]:
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(estimator= Temp_LSTM_model,
                                   param_distributions = hyperparam_list,
                                   scoring= make_scorer(mean_squared_error, greater_is_better=False),
                                   cv = 2)

In [27]:
print('Date Time 컬럼 데이터의 type : ', type(weather_data['Date Time'][0]))

Date Time 컬럼 데이터의 type :  <class 'str'>


In [28]:
for i in range(len(weather_data)):
    if weather_data['Date Time'][i][6:10] =='2016':
        print('2016년 데이터는 ', str(i), '번째 행부터 시작입니다.')
        break

2016년 데이터는  368291 번째 행부터 시작입니다.


In [29]:
weather_data = weather_data.iloc[368291:]
weather_data = weather_data.reset_index(drop=True)
weather_data

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2016 00:00:00,999.08,-0.01,273.22,-0.44,96.90,6.10,5.91,0.19,3.69,5.92,1271.32,1.16,2.04,192.4
1,01.01.2016 00:10:00,999.03,0.01,273.25,-0.41,97.00,6.11,5.93,0.18,3.70,5.94,1271.16,1.01,2.12,211.6
2,01.01.2016 00:20:00,999.07,0.06,273.29,-0.36,97.00,6.13,5.95,0.18,3.71,5.96,1270.97,0.80,1.52,203.8
3,01.01.2016 00:30:00,999.09,0.07,273.30,-0.36,96.90,6.14,5.95,0.19,3.71,5.96,1270.93,0.77,1.64,184.2
4,01.01.2016 00:40:00,999.09,-0.05,273.18,-0.50,96.80,6.09,5.89,0.19,3.68,5.90,1271.54,0.84,1.92,200.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52255,31.12.2016 23:20:00,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
52256,31.12.2016 23:30:00,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
52257,31.12.2016 23:40:00,999.82,-3.16,270.01,-8.21,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2
52258,31.12.2016 23:50:00,999.81,-4.23,268.94,-8.53,71.80,4.46,3.20,1.26,1.99,3.20,1293.56,1.49,2.16,225.8


In [30]:
weather_data = weather_data.iloc[:,1:]
weather_data

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,999.08,-0.01,273.22,-0.44,96.90,6.10,5.91,0.19,3.69,5.92,1271.32,1.16,2.04,192.4
1,999.03,0.01,273.25,-0.41,97.00,6.11,5.93,0.18,3.70,5.94,1271.16,1.01,2.12,211.6
2,999.07,0.06,273.29,-0.36,97.00,6.13,5.95,0.18,3.71,5.96,1270.97,0.80,1.52,203.8
3,999.09,0.07,273.30,-0.36,96.90,6.14,5.95,0.19,3.71,5.96,1270.93,0.77,1.64,184.2
4,999.09,-0.05,273.18,-0.50,96.80,6.09,5.89,0.19,3.68,5.90,1271.54,0.84,1.92,200.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52255,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
52256,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
52257,999.82,-3.16,270.01,-8.21,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2
52258,999.81,-4.23,268.94,-8.53,71.80,4.46,3.20,1.26,1.99,3.20,1293.56,1.49,2.16,225.8


In [42]:
def extract_inputoutput(dataframe, lookback_time=2, predict_time=1):
    dfx = pd.DataFrame()
    dfy = pd.DataFrame()

    for i in range(len(dataframe) - (lookback_time - 1) - (predict_time)):
        if i % 1000 == 0:
            print(i)
        rowx = []
        for timestep in range(lookback_time):
            dfRename = dataframe.iloc[[i+timestep]]
            dfRename.index=[i]
            rowx.append(dfRename)
        rowx = pd.concat(rowx, axis=1)
        dfx = pd.concat([dfx, rowx])
        rowy = []
        rowy = pd.DataFrame([dataframe['T (degC)'][i+lookback_time]])
        dfy = pd.concat([dfy, rowy], ignore_index=True)

    print('X, Y 데이터 분류 완료!')
    return dfx, dfy

In [43]:
x, t = extract_inputoutput(weather_data)

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000
25000
26000
27000
28000
29000
30000
31000
32000
33000
34000
35000
36000
37000
38000
39000
40000
41000
42000
43000
44000
45000
46000
47000
48000
49000
50000
51000
52000
X, Y 데이터 분류 완료!


In [44]:
x

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),...,rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,999.08,-0.01,273.22,-0.44,96.90,6.10,5.91,0.19,3.69,5.92,...,97.00,6.11,5.93,0.18,3.70,5.94,1271.16,1.01,2.12,211.6
1,999.03,0.01,273.25,-0.41,97.00,6.11,5.93,0.18,3.70,5.94,...,97.00,6.13,5.95,0.18,3.71,5.96,1270.97,0.80,1.52,203.8
2,999.07,0.06,273.29,-0.36,97.00,6.13,5.95,0.18,3.71,5.96,...,96.90,6.14,5.95,0.19,3.71,5.96,1270.93,0.77,1.64,184.2
3,999.09,0.07,273.30,-0.36,96.90,6.14,5.95,0.19,3.71,5.96,...,96.80,6.09,5.89,0.19,3.68,5.90,1271.54,0.84,1.92,200.1
4,999.09,-0.05,273.18,-0.50,96.80,6.09,5.89,0.19,3.68,5.90,...,97.10,6.14,5.96,0.18,3.72,5.97,1270.93,0.33,0.84,159.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52253,1000.21,-3.76,269.39,-7.95,72.50,4.62,3.35,1.27,2.09,3.35,...,72.60,4.56,3.31,1.25,2.06,3.31,1292.41,0.56,1.00,202.6
52254,1000.11,-3.93,269.23,-8.09,72.60,4.56,3.31,1.25,2.06,3.31,...,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
52255,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,...,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
52256,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,...,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2


In [45]:
t

,0
0,0.06
1,0.07
2,-0.05
3,0.07
4,-0.05
...,...
52253,-4.05
52254,-3.35
52255,-3.16
52256,-4.23


In [46]:
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size=0.2, shuffle=False)

print('x_train shape : ', x_train.shape)
print('t_train shape : ', t_train.shape)
print('x_test shape : ', x_test.shape)
print('t_test shape : ', t_test.shape)

x_train shape :  (41806, 28)
t_train shape :  (41806, 1)
x_test shape :  (10452, 28)
t_test shape :  (10452, 1)


In [47]:
timesteps = 2
feature = 14

x_train = np.array(x_train)
x_train = x_train.reshape(x_train.shape[0], timesteps, feature)

x_test = np.array(x_test)
x_test = x_test.reshape(x_test.shape[0], timesteps, feature)

t_train = np.array(t_train)
t_test = np.array(t_test)

print('reshape 후 x_train shape :', x_train.shape)
print('t_train shape : ', t_train.shape)
print('reshape 후 x_test shape :', x_test.shape)
print('t_test shape : ', t_test.shape)

reshape 후 x_train shape : (41806, 2, 14)
t_train shape :  (41806, 1)
reshape 후 x_test shape : (10452, 2, 14)
t_test shape :  (10452, 1)


In [48]:
hyperparam_list = {'epochs' : [10, 20],
                   'batch_size' : [16, 32]}

print('사용할 hyperparameter 목록 : ', hyperparam_list)

사용할 hyperparameter 목록 :  {'epochs': [10, 20], 'batch_size': [16, 32]}


In [49]:
def lstm_model():
    cell_size = 128
    timesteps = 2
    feature = 14

    model = Sequential(name='Temp_LSTM')

    model.add(LSTM(cell_size, input_shape=(timesteps, feature), return_sequences=True))
    model.add(LSTM(cell_size))
    model.add(Dropout(0.1))

    model.add(Dense(1))
    model.compile(loss=mse, optimizer=Adam(learning_rate=0.001),
                  metrics=[RootMeanSquaredError()])
    
    return model

In [50]:
Temp_LSTM_model = KerasRegressor(build_fn = lstm_model)

from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(estimator = Temp_LSTM_model,
                                   param_distributions=hyperparam_list,
                                   scoring = make_scorer(mean_squared_error, greater_is_better=False),
                                   cv = 2)
print('random search : ', random_search, sep='\n')

random search : 
RandomizedSearchCV(cv=2,
                   estimator=KerasRegressor(build_fn=<function lstm_model at 0x12a63f240>),
                   param_distributions={'batch_size': [16, 32],
                                        'epochs': [10, 20]},
                   scoring=make_scorer(mean_squared_error, greater_is_better=False, response_method='predict'))


In [51]:
random_search_result = random_search.fit(x_train, t_train, verbose = 1)

Epoch 1/10


/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 4 is smaller than n_iter=10. Running 4 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1307/1307 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 17.0057 - root_mean_squared_error: 4.1238
Epoch 2/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 4.3668 - root_mean_squared_error: 2.0897
Epoch 3/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 3.4777 - root_mean_squared_error: 1.8649
Epoch 4/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 2.7951 - root_mean_squared_error: 1.6719
Epoch 5/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 2.3118 - root_mean_squared_error: 1.5205
Epoch 6/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 2.0387 - root_mean_squared_error: 1.4278
Epoch 7/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 1.8329 - root_mean_squared_error: 1.3539
Epoch 8/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 1.5268 - root_mean_squared_error: 1.2356
Epoch 9/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 1.5845 - root_mean_squared_error: 1.2588
Epoch 10/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 1.6564 - root_mean_s

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1307/1307 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 5.2461 - root_mean_squared_error: 2.2904 
Epoch 2/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.8762 - root_mean_squared_error: 0.9361
Epoch 3/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7415 - root_mean_squared_error: 0.8611
Epoch 4/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7437 - root_mean_squared_error: 0.8624
Epoch 5/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7447 - root_mean_squared_error: 0.8630
Epoch 6/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7311 - root_mean_squared_error: 0.8550
Epoch 7/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.6952 - root_mean_squared_error: 0.8338
Epoch 8/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.6159 - root_mean_squared_error: 0.7848
Epoch 9/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.7234 - root_mean_squared_error: 0.8505
Epoch 10/10
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - loss: 0.6581 - root_mean_s

...

654/654 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 41.8900 - root_mean_squared_error: 6.4722
Epoch 2/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3580 - root_mean_squared_error: 5.5998
Epoch 3/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3826 - root_mean_squared_error: 5.6020
Epoch 4/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.4546 - root_mean_squared_error: 5.6084
Epoch 5/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3974 - root_mean_squared_error: 5.6033
Epoch 6/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.4308 - root_mean_squared_error: 5.6063
Epoch 7/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 31.3819 - root_mean_squared_error: 5.6020
Epoch 8/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3612 - root_mean_squared_error: 5.6001
Epoch 9/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3755 - root_mean_squared_error: 5.6014
Epoch 10/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 31.3924 - root_mean_squared_erro

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


654/654 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 6.2227 - root_mean_squared_error: 2.4945 
Epoch 2/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7408 - root_mean_squared_error: 0.8607
Epoch 3/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5032 - root_mean_squared_error: 0.7094
Epoch 4/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4709 - root_mean_squared_error: 0.6862
Epoch 5/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4780 - root_mean_squared_error: 0.6914
Epoch 6/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4330 - root_mean_squared_error: 0.6581
Epoch 7/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4287 - root_mean_squared_error: 0.6547
Epoch 8/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5376 - root_mean_squared_error: 0.7332
Epoch 9/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5215 - root_mean_squared_error: 0.7222
Epoch 10/20
654/654 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.4926 - root_mean_squared_error: 0.7018

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/scikeras/wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


2613/2613 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - loss: 6.1132 - root_mean_squared_error: 2.4725 
Epoch 2/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.3588 - root_mean_squared_error: 1.1657
Epoch 3/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.3343 - root_mean_squared_error: 1.1551
Epoch 4/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.1213 - root_mean_squared_error: 1.0589
Epoch 5/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.0576 - root_mean_squared_error: 1.0284
Epoch 6/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.2033 - root_mean_squared_error: 1.0970
Epoch 7/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.1976 - root_mean_squared_error: 1.0944
Epoch 8/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.0844 - root_mean_squared_error: 1.0413
Epoch 9/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.1111 - root_mean_squared_error: 1.0541
Epoch 10/20
2613/2613 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - loss: 1.0125 - root_mean_s

In [52]:
print("Random Search에서의 최고의 하이퍼파라미터 조합 :", random_search_result.best_params_)

Random Search에서의 최고의 하이퍼파라미터 조합 : {'epochs': 20, 'batch_size': 16}


In [53]:
temp_lstm = lstm_model()
temp_lstm.summary()

/Users/yuseunghwan/workspace/univ_dev/ml2/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "Temp_LSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_38 (LSTM)                  │ (None, 2, 128)         │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_39 (LSTM)                  │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 204,929 (800.50 KB)

 Trainable params: 204,929 (800.50 KB)

 Non-trainable params: 0 (0.00 B)

In [54]:
x_train, x_validation, t_train, t_validation = train_test_split(x_train, t_train, test_size=0.5, shuffle=False)

print('x_train shape : ', x_train.shape)
print('t_train shape : ', t_train.shape)
print('x_validation shape : ', x_validation.shape)
print('t_validation shape : ', t_validation.shape)

x_train shape :  (20903, 2, 14)
t_train shape :  (20903, 1)
x_validation shape :  (20903, 2, 14)
t_validation shape :  (20903, 1)


In [56]:
batch_size = random_search.best_params_['batch_size']
epoch = random_search.best_params_['epochs']

early_stopping = EarlyStopping(monitor='val_loss', mode='min', patience=epoch*0.3)
temp_lstm.fit(x_train, t_train, validation_data=(x_validation, t_validation),
              batch_size=batch_size, epochs=epoch, callbacks=[early_stopping])

Epoch 1/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 6.6947 - root_mean_squared_error: 2.5874 - val_loss: 6.5664 - val_root_mean_squared_error: 2.5625
Epoch 2/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.9353 - root_mean_squared_error: 0.9671 - val_loss: 4.5891 - val_root_mean_squared_error: 2.1422
Epoch 3/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7756 - root_mean_squared_error: 0.8807 - val_loss: 3.8514 - val_root_mean_squared_error: 1.9625
Epoch 4/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.7600 - root_mean_squared_error: 0.8718 - val_loss: 3.9388 - val_root_mean_squared_error: 1.9846
Epoch 5/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.6898 - root_mean_squared_error: 0.8306 - val_loss: 3.6209 - val_root_mean_squared_error: 1.9029
Epoch 6/20
1307/1307 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.6969 - root_mean_squared_error: 0.8348 - val_loss: 2.5151 - val_root_mean_squared_error: 1.5859
Epoch 7/20
1307/1307 ━━━━━━━━━━━━━━━━━━━

In [58]:
pred = temp_lstm.predict(x_test)

for i in range(1, 10):
    print('온도(T(degC)) 예측 : ', round(pred[i][0], 2), '/ 정답 : ', round(t_test[i][0], 2))

327/327 ━━━━━━━━━━━━━━━━━━━━ 0s 562us/step
온도(T(degC)) 예측 :  5.22 / 정답 :  5.94
온도(T(degC)) 예측 :  5.18 / 정답 :  6.16
온도(T(degC)) 예측 :  5.42 / 정답 :  6.43
온도(T(degC)) 예측 :  5.74 / 정답 :  6.58
온도(T(degC)) 예측 :  5.89 / 정답 :  6.93
온도(T(degC)) 예측 :  6.27 / 정답 :  7.21
온도(T(degC)) 예측 :  6.6 / 정답 :  7.38
온도(T(degC)) 예측 :  6.8 / 정답 :  7.56
온도(T(degC)) 예측 :  7.08 / 정답 :  8.14


In [59]:
loss, rmse = temp_lstm.evaluate(x_test, t_test, verbose=1)
print('test loss(MSE) : ', round(loss, 6))
print('test RMSE : ', round(rmse, 2))

327/327 ━━━━━━━━━━━━━━━━━━━━ 0s 675us/step - loss: 0.3694 - root_mean_squared_error: 0.6078
test loss(MSE) :  0.369378
test RMSE :  0.61
